# TOU의 가계 환류와 REN 생산지원 비교
GAMS Run_Master.gms 실행 후 이 노트북을 위에서부터 실행합니다.
BAU, TOU, 하나의 TOU_REN_HH 정책군을 비교합니다. a=0은 HH, a=1은 REN입니다.
r=0은 기존 TOU 해를 재사용합니다. 격자에는 r=0 조합도 표시하지만 별도의 정책으로 세지 않습니다.
정책 재원은 환류 없는 TOU의 양수 소비 wedge 수입으로 고정합니다. 실제 한전 이익이 아닙니다.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Run_Master.gms').is_file())
ROOT = PROJECT/'Micro'
sys.path.insert(0,str(ROOT))
from tou_policy_results import load_policy_results
GAMS_DIR = Path(r'C:\GAMS\40')
GDX_FILE = PROJECT/'TOU_recycling_results.gdx'
REGION = '01_KOR'
SELECTED_RATE = 'r100'
SELECTED_MIX = 'a50'
OUTPUT_DIR = ROOT/'policy_analysis'/'cge'
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
x = load_policy_results(GDX_FILE,GAMS_DIR)
macro=x['macro']
selected_policy_id=SELECTED_RATE+'_'+SELECTED_MIX


## 1. 대표 정책 비교
후생은 가계 구매자가격을 쓰는 Stone–Geary EV입니다. 기본소비는 고정되어 있습니다.
EV는 변화액이며 EV/기준소비지출은 %입니다. EV의 BAU 대비 증가율은 정의하지 않습니다.
23_eREN은 재생 및 기타 발전이라는 모형 집계입니다. 원자력·화석발전은 지원에서 제외합니다.
생산세 감면은 정태적인 생산지원입니다. 신규 설비 건설과 장기 투자 효과를 뜻하지 않습니다.


In [ ]:
k=macro.loc[macro.region.eq(REGION)].copy()
summary=k.pivot(index=['policy_id','scenario','rate','mix','recycling_fraction'],columns='macro_item',values='value').reset_index()
# ren_fraction은 기준 시나리오에서 결측이므로 pivot 인덱스에 넣지 않습니다.
summary=summary.merge(x['grid'][['policy_id','ren_fraction']],on='policy_id',how='left')
base0=summary.loc[summary.policy_id.eq('BAU')].iloc[0]
for name in ['GDP_REAL','CTH_DEFLATED','HH_ELEC_GWH','CPI']:
    summary[name+'_change_pct']=(summary[name]/base0[name]-1)*100
summary['CO2_change_Mt']=(summary.CO2_TOTAL_KT-base0.CO2_TOTAL_KT)/1000
summary['CO2_change_pct']=(summary.CO2_TOTAL_KT/base0.CO2_TOTAL_KT-1)*100
summary['power_CO2_change_Mt']=(summary.CO2_POWER_COMB_KT-base0.CO2_POWER_COMB_KT)/1000
summary['EV_billion_USD_conditional']=summary.EV*x['money_to_billion_usd']
ids=list(dict.fromkeys(['BAU','TOU',SELECTED_RATE+'_a00',selected_policy_id,SELECTED_RATE+'_a100']))
headline=summary.set_index('policy_id').reindex(ids)
display(headline[['scenario','EV_PCT_CTH','GDP_REAL_change_pct','CO2_change_Mt','HH_ELEC_GWH_change_pct']])


## 2. 환류율–REN 배분비율 격자
같은 r에서 a를 바꾸면 전체 예산은 같고, HH와 REN의 몫만 달라집니다.
같은 a에서 r을 바꾸면 지원예산의 크기가 달라집니다.
표의 r=0 행은 모두 동일합니다. 아래 전체 결과에는 모든 조합을 보존합니다.


In [ ]:
policy_grid=summary.loc[summary.rate.ne('base')]
display(policy_grid.pivot(index='recycling_fraction',columns='ren_fraction',values='EV_PCT_CTH'))
display(policy_grid.pivot(index='recycling_fraction',columns='ren_fraction',values='CO2_change_Mt'))


## 3. 재원 사용과 기존 재정계정의 잔여 수입
reference_retained=(1-r)×고정 기준재원은 사전 배분액입니다.
fiscal_residual_policy=사후 wedge 수입−HH 예산−REN 예산입니다.
REN으로 전력가격·소비가 변하므로 사후 수입은 기준재원과 다릅니다. 잔여 수입은 음수일 수도 있습니다.
정부저축 SG는 다른 조세·가격 변화까지 포함해 내생적으로 결정됩니다.
REN 비용은 생산세 수입 감소에 반영됩니다. 정부저축에서 같은 금액을 다시 빼지 않습니다.
아래 값은 모형 화폐 단위이며 한전의 영업이익·부채상환액이 아닙니다.


In [ ]:
budgets=x['budgets']
budget_view=budgets.loc[budgets.region.eq(REGION)&budgets.policy_id.isin(ids)]
display(budget_view[['policy_id','budget_total','budget_hh','budget_ren','ren_support_actual','reference_retained','fiscal_residual_policy']])


## 4. 발전원과 배출 구성
발전은 기준가격 수량과 BAU 대비 변화율을 사용합니다. 발전원별 기준 GWh가 없어 물리적 발전비중은 추정하지 않습니다.
총배출=산업 연소+비에너지 사용의 저장탄소 조정 배출+가계 직접연소.
발전부문 연소는 산업 연소의 부분집합입니다. 구매전력 간접배출을 다시 더하지 않습니다.
포괄적 산업공정 배출·시간대 급전·시간대 탄소계수는 포함하지 않습니다.


In [ ]:
generation=x['generation']
gen_view=generation.loc[generation.region.eq(REGION)&generation.policy_id.isin(ids)]
display(gen_view.pivot(index='industry',columns='policy_id',values='change_pct').reindex(columns=ids))
co2_items=['CO2_INDUSTRY_COMB_KT','CO2_NONENERGY_KT','CO2_HH_DIRECT_KT','CO2_TOTAL_KT','CO2_POWER_COMB_KT']
co2=k.loc[k.macro_item.isin(co2_items)&k.policy_id.isin(ids)].copy()
co2['change_Mt']=co2.change/1000
display(co2.pivot(index='macro_item',columns='policy_id',values='change_Mt').reindex(columns=ids))
global_co2=macro.loc[macro.macro_item.eq('CO2_TOTAL_KT')].groupby(['policy_id','scenario','rate','mix'],as_index=False)[['value','bau_value','change']].sum()
global_co2['change_Mt']=global_co2.change/1000


## 5. 가계 시간대별 수요와 미시분석 연결
가계 GWh는 기존 EHI_ELEC 변환계수에 의한 추계입니다. 기준 ktoe에 11.63을 곱합니다.
CGE의 대표가계 EV와 미시 가구별 순전기요금은 다른 범위이며 합산하지 않습니다.
다음 미시 노트북은 각 정책점의 시간대 가격배율과 HH 몫을 읽습니다.


In [ ]:
loads=x['loads'].copy()
q0=loads.loc[loads.policy_id.eq('BAU')].groupby('region').value.sum()
gwh0=macro.loc[macro.policy_id.eq('BAU')&macro.macro_item.eq('HH_ELEC_GWH')].set_index('region').value
loads['load_GWh']=loads.value*loads.region.map(gwh0/q0)
display(loads.loc[loads.region.eq(REGION)&loads.policy_id.isin(ids)].pivot(index='block',columns='policy_id',values='change_pct').reindex(columns=ids))


In [ ]:
tables=dict(headline=headline.reset_index(),policy_summary=summary,macro_all_regions=macro,
    budget_all_regions=budgets,generation_all_regions=generation,emissions_sector_all_regions=x['emissions'],
    global_CO2=global_co2,household_load_all_regions=loads)
for name,table in tables.items():table.to_csv(OUTPUT_DIR/(name+'.csv'),index=False,encoding='utf-8-sig')
metadata=dict(gdx_file=str(GDX_FILE),gdx_modified=GDX_FILE.stat().st_mtime,region=REGION,
    selected_rate=SELECTED_RATE,selected_mix=SELECTED_MIX,
    ren_scope='23_eREN: renewable and other generation; no nuclear or fossil support',
    budget_rule='frozen positive no-recycling TOU wedge receipts; not utility profit',
    model_money_to_billion_usd_conditional=x['money_to_billion_usd'],
    caveats=['Static production support, not capital investment','No hourly dispatch',
             'Government fiscal residual is not utility profit','CGE EV is not micro EV'])
(OUTPUT_DIR/'definitions.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('저장 위치:',OUTPUT_DIR)
